# Смеси распределений, интенсивность отказов и случайные суммы [5 баллов]
**Вариант 1**

### Импорты и настройка

В этой ячейке собраны все необходимые импорты, генератор псевдослучайных чисел и константы задачи.

**Ничего менять не нужно — просто запусти ячейку.**

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np

# RNG для воспроизводимости
RNG = np.random.default_rng(42)

# Константы части I
ALPHA = 3.0                                    # параметр формы Gamma(α, β)
BETA = 2.0                                     # параметр скорости (rate) Gamma(α, β)
T_GRID = np.linspace(0.0, 4.0, 80)             # сетка времени t для оценки λ(t)
DT = 0.05                                      # шаг окна [t, t + Δt) для оценки плотности
N_MIX = 200_000                                # объём выборки для моделирования смеси

# Константы части II
LAMBDA_EXP = 2.0                               # интенсивность времени одного шага X_i ~ Exp(λ)
P_GEOM = 0.25                                  # параметр геометрического распределения K ~ Geom(p)
N_SUM = 100_000                                # объём выборки для случайных сумм

print('Импорты загружены.')
print(f'Часть I:  Λ ~ Gamma(α={ALPHA}, β={BETA}), N_MIX = {N_MIX:,}')
print(f'Часть II: X_i ~ Exp(λ={LAMBDA_EXP}), K ~ Geom(p={P_GEOM}), N_SUM = {N_SUM:,}')

## Часть I. Непрерывная смесь и функция интенсивности отказов [2 балла]

Инженеры надёжности (SRE) изучают время безотказной работы серверных узлов $T > 0$ с плотностью $f(t)$ и функцией распределения $F(t) = \mathbb{P}(T \le t)$

Обычная плотность $f(t)$ описывает частоту отказов по отношению ко всем запущенным в момент $t=0$ серверам, поэтому на больших временах $f(t)$ всегда стремится к нулю просто потому, что работающих узлов остаётся мало. Чтобы понять, изнашивается ли сам работающий узел, нужно смотреть на условную вероятность отказа в малом интервале $[t, t + \Delta t)$ при условии, что к моменту $t$ узел ещё жив ($T > t$):

$$\mathbb{P}(t \le T < t + \Delta t \mid T > t) = \frac{\mathbb{P}(t \le T < t + \Delta t)}{\mathbb{P}(T > t)} \approx \frac{f(t)\,\Delta t}{1 - F(t)}$$

Коэффициент пропорциональности при $\Delta t$ называется **функцией интенсивности** (hazard rate):

$$\lambda(t) = \frac{f(t)}{1 - F(t)}$$

- Если $\lambda(t)$ растёт со временем - элементы изнашиваются (чем дольше проработал, тем выше риск отказа в следующую секунду)
- Если $\lambda(t) = \lambda_0 = \text{const}$ - система обладает **свойством отсутствия памяти**: прошлое время работы никак не влияет на текущий риск отказа. Этим свойством характеризуется экспоненциальное распределение  $T \sim \text{Exp}(\lambda_0)$
- В реальном дата-центре серверы поступают из разных партий, поэтому индивидуальная интенсивность сбоев является случайной:
  $$\Lambda \sim \text{Gamma}(\alpha, \beta), \qquad T \mid \Lambda \sim \text{Exp}(\Lambda),$$
  где плотность $\Lambda$ равна $g(\lambda) = \frac{\beta^\alpha}{\Gamma(\alpha)}\lambda^{\alpha-1}e^{-\beta\lambda}$ ($\lambda > 0$). Каждый отдельный сервер в таком парке не изнашивается ($\Lambda_i = \text{const}$). Проверим, что происходит с интенсивностью отказов всего парка $\lambda(t)$

### Задача 1.1. Генерация смеси и оценка функции интенсивности [1 балл]

1. Реализуй функцию `sample_gamma_exp_mixture`, которая генерирует массив индивидуальных интенсивностей $\Lambda_i \sim \text{Gamma}(\alpha, \beta)$ и массив времён жизни $T_i \mid \Lambda_i \sim \text{Exp}(\Lambda_i)$ [0,5 балла]
2. Реализуй функцию `empirical_hazard_rate`, которая оценивает $\hat{\lambda}(t)$ на сетке `t_grid` [0,5 балла]

Реализованные функции должны быть векторизованными

In [ ]:
def sample_gamma_exp_mixture(
    alpha: float,
    beta: float,
    size: int,
    rng: np.random.Generator = RNG,
) -> tuple[np.ndarray, np.ndarray]:
    """Генерация смеси

    Parameters
    ----------
    alpha : float
        Параметр формы α гамма-распределения
    beta : float
        Параметр скорости (rate) β гамма-распределения (scale = 1 / beta)
    size : int
        Объём выборки
    rng : np.random.Generator
        Генератор псевдослучайных чисел

    Returns
    -------
    lambdas : np.ndarray, shape (size,)
        Индивидуальные интенсивности Λ_i ~ Gamma(α, β)
    samples : np.ndarray, shape (size,)
        Времена до отказа T_i | Λ_i ~ Exp(Λ_i)
    """
    lambdas = None  # TODO
    samples = None  # TODO
    return lambdas, samples


def empirical_hazard_rate(
    samples: np.ndarray,
    t_grid: np.ndarray,
    dt: float = DT,
) -> np.ndarray:
    """Численная оценка функции интенсивности отказов λ(t) = f(t) / (1 - F(t))

    Parameters
    ----------
    samples : np.ndarray, shape (n_samples,)
        Выборка времён жизни T
    t_grid : np.ndarray, shape (n_grid,)
        Сетка моментов времени t
    dt : float
        Ширина полуинтервала [t, t + dt) для оценки плотности f(t)

    Returns
    -------
    hazard : np.ndarray, shape (n_grid,)
        Оценка интенсивности отказов в точках t_grid
    """
    hazard = None    # TODO
    return hazard

*Запусти ячейку как есть — ничего менять не нужно.*

In [ ]:
# Техническая проверка: запусти ячейку как есть
_lam_test, _t_test = sample_gamma_exp_mixture(ALPHA, BETA, 10_000, rng=np.random.default_rng(0))
assert _lam_test is not None and _t_test is not None, 'Реализуй sample_gamma_exp_mixture'
assert _lam_test.shape == (10_000,) and _t_test.shape == (10_000,), 'Неверная форма массивов'
assert np.all(_lam_test > 0) and np.all(_t_test > 0), 'Значения должны быть положительными'
assert abs(_lam_test.mean() - ALPHA / BETA) < 0.05, 'Среднее Λ не соответствует Gamma(α, β) - проверь параметр scale'

_calib_samples = np.random.default_rng(0).exponential(scale=1.0, size=100_000)
_calib_hazard = empirical_hazard_rate(_calib_samples, np.linspace(0.0, 2.0, 20), dt=0.05)
assert _calib_hazard is not None and _calib_hazard.shape == (20,), 'Неверная форма результата empirical_hazard_rate'
assert np.allclose(_calib_hazard, 1.0, atol=0.15), 'На выборке Exp(1) интенсивность должна быть близка к 1.0'

print('Технические проверки пройдены')

### Задача 1.2. Численный эксперимент: однородный парк против гамма-смеси [0,5 балла]

Проведи численный эксперимент для объёма выборки `N_MIX`:

1. Вычисли теоретическое математическое ожидание интенсивности $\lambda_0 = \mathbb{E}[\Lambda]$
2. С помощью `sample_gamma_exp_mixture` сгенерируй выборку $(\Lambda_i, T_i)_{i=1}^N$ для неоднородного парка серверов с параметрами `ALPHA`, `BETA`
3. Сгенерируй выборку $T^{(0)}_1, \dots, T^{(0)}_N \sim \text{Exp}(\lambda_0)$ для однородного парка серверов, в котором у каждого узла интенсивность отказов фиксирована и равна средней: $\Lambda_i \equiv \lambda_0$
4. Вычисли выборочное среднее и выборочное стандартное отклонение времени жизни для обеих выборок ($T$ и $T^{(0)}$), а также рассчитай функции интенсивности отказов $\hat{\lambda}_{\text{mix}}(t)$ и $\hat{\lambda}_{\text{pure}}(t)$ для первого и последнего моментов времени

In [ ]:
# Твой код здесь

### Задача 1.3. Визуализация [0,5 балла]

Построй графики, которые дают наглядный ответ на два вопроса:

1. Как ведёт себя во времени эмпирическая интенсивность отказов $\hat{\lambda}(t)$ для неоднородной гамма-смеси по сравнению с однородным парком  и с аналитической кривой интенсивности гамма-смеси $\lambda_{\text{theor}}(t) = \frac{\alpha}{\beta + t}$?
2. Как меняется распределение индивидуальных интенсивностей $\Lambda$ среди выживших к моменту $t$ узлов при увеличении времени наблюдения $t$?

In [ ]:
# Твой код визуализации здесь

## Часть II. Случайные суммы [2 балла]


При передаче сетевого пакета через цепочку маршрутизаторов время одной попытки пересылки распределено экспоненциально: $X_i \sim \text{Exp}(\lambda)$

Сравним два режима работы протокола с **одинаковым средним числом попыток** $m = \mathbb{E}[K] = \frac{1}{p}$:
- **Фиксированное число попыток ($K \equiv m$):** пакет всегда проходит ровно $m$ этапов, а суммарное время равно $S_m = X_1 + \dots + X_m$
- **Случайное число попыток до первого успеха ($K \sim \text{Geom}(p)$):** после каждой попытки передача завершается успешно с вероятностью $p$ или повторяется с вероятностью $1 - p$ ($K \in \{1, 2, 3, \dots\}$), а суммарное время равно случайной сумме:
  $$S_K = \sum_{i=1}^K X_i$$

### Задача 2.1. Генератор случайной суммы [1 балл]

Реализуй функцию `sample_geometric_exp_sum`, которая генерирует выборку случайных сумм $S_K = \sum_{i=1}^K X_i$ объёма `size` 

Реализация должна быть векторизованной

In [ ]:
def sample_geometric_exp_sum(
    lam: float,
    p_geom: float,
    size: int,
    rng: np.random.Generator = RNG,
) -> tuple[np.ndarray, np.ndarray]:
    """Генерация геометрической суммы экспоненциальных величин.

    Parameters
    ----------
    lam : float
        Параметр интенсивности λ отдельных слагаемых X_i ~ Exp(λ)
    p_geom : float
        Вероятность успеха p геометрического числа слагаемых K ~ Geom(p) на {1, 2, ...}
    size : int
        Объём выборки (количество случайных сумм S_K)
    rng : np.random.Generator
        Генератор псевдослучайных чисел

    Returns
    -------
    k_counts : np.ndarray, shape (size,)
        Массив случайных количеств слагаемых K_j
    s_random : np.ndarray, shape (size,)
        Массив случайных сумм S_j = X_{j,1} + ... + X_{j,K_j}
    """
    k_counts = None      # TODO
    s_random = None      # TODO
    return k_counts, s_random

*Запусти ячейку как есть — ничего менять не нужно*

In [ ]:
# Техническая проверка: запусти ячейку как есть
_t0 = time.perf_counter()
_k_test, _s_test = sample_geometric_exp_sum(LAMBDA_EXP, P_GEOM, N_SUM, rng=np.random.default_rng(42))
_elapsed = time.perf_counter() - _t0

assert _k_test is not None and _s_test is not None, 'Реализуй sample_geometric_exp_sum'
assert _k_test.shape == (N_SUM,) and _s_test.shape == (N_SUM,), 'Неверная форма выходных массивов'
assert np.all(_k_test >= 1) and np.all(_s_test > 0), 'Некорректные значения K или S'

print(f'Техническая проверка пройдена: {N_SUM:,} случайных сумм сгенерированы за {_elapsed * 1000:.1f} мс')

### Задача 2.2. Численный эксперимент: случайная сумма $S_K$ против фиксированной суммы $S_m$ [0,5 балла]

Согласно теореме Реньи, геометрическая сумма независимых величин $X_i \sim \text{Exp}(\lambda)$ при $K \sim \text{Geom}(p)$ сохраняет экспоненциальный тип распределения с параметром $\mu = \lambda \cdot p$. Проверим это численно для объёма выборки `N_SUM`:

1. Сгенерируй выборку случайных сумм $S_K$ с параметрами `LAMBDA_EXP` и `P_GEOM` с помощью `sample_geometric_exp_sum`
2. Сгенерируй выборку детерминированных сумм $S_m = X_1 + \dots + X_m$ того же объёма `N_SUM` при фиксированном числе слагаемых $m$
3. Вычисли и сопоставь выборочное среднее и выборочное стандартное отклонение для $S_K$ и для $S_m$ с теоретическим значением 
4. Используя функцию `empirical_hazard_rate` из части I, вычисли эмпирическую интенсивность отказов на сетке `T_GRID` для $S_K$ и для $S_m$. Выведи на экран значения для первого и последнего моментов времени

In [ ]:
# Твой код здесь

### Задача 2.3. Визуализация [0,5 балла]

Построй графики, которые наглядно отвечают на следующие вопросы:

1. Как выглядят гистограмма и эмпирическая функция распределения случайной суммы $S_K$ в сравнении с теоретическим распределением $\text{Exp}(\lambda p)$ и с распределением фиксированной суммы $S_m$?
2. Как ведёт себя во времени функция интенсивности отказов $\hat{\lambda}(t)$ для случайной суммы $S_K$ и для фиксированной суммы $S_m$?

In [ ]:
# Твой код визуализации здесь

## Задача 3. Выводы [1 балл]

Сформулируй содержательные выводы по двум проведенным экспериментам. Объясни наблюдаемые явления

**1.** ...

**2.** ...

**...**